In [0]:
-- Calculate defaulted loan percentage by region
SELECT Region,ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100,2) AS Defaulted_pct
FROM loan_default_workspace.default.loans_silver
GROUP BY Region
ORDER BY Defaulted_pct DESC;

Region,Defaulted_pct
North-East,30.45
central,27.54
south,26.63
North,22.51


In [0]:
-- Calculate overall defaulted loan percentage
SELECT ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100,2) AS Defaulted_pct
FROM loan_default_workspace.default.loans_silver

Defaulted_pct
24.64


In [0]:
%sql
-- Calculate loan count and defaulted loan percentage by credit score bucket
SELECT COUNT(*) AS loan_count,
(CASE 
WHEN Credit_Score <600 THEN 'LOW'
WHEN Credit_Score BETWEEN 600 and 700 THEN 'MEDIUM'
WHEN Credit_Score > 700 THEN 'HIGH'
END) AS Credit_bucket,
ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100,2) AS Defaulted_pct
FROM loan_default_workspace.default.loans_silver
GRoup by Credit_bucket
ORDER BY Defaulted_pct DESC
;

loan_count,Credit_bucket,Defaulted_pct
73877,HIGH,24.73
37190,LOW,24.63
37603,MEDIUM,24.49


In [0]:
%sql
-- Calculate loan count and defaulted loan percentage by DTI bucket
Select 
CASE
    WHEN dtir1 < 30 THEN 'Low (< 30)'
    WHEN dtir1 BETWEEN 30 AND 45 THEN 'Medium (30–45)'
    ELSE 'High (> 45)'
 END AS dti_bucket,
 COUNT(*) AS loan_count,
 ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100,2) AS Defaulted_pct

from 
  loan_default_workspace.default.loans_silver
  Group by 1
  order by 3 desc

dti_bucket,loan_count,Defaulted_pct
Medium (30–45),96505,26.49
High (> 45),28453,25.25
Low (< 30),23712,16.4


In [0]:
%sql
-- Calculate loan count and defaulted loan percentage for 'HIGH DTI + HIGH LTV' risk group vs everyone else
Select 
CASE
WHEN dtir1 > 30 AND LTV >80 THEN 'HIGH DTI + HIGH LTV'
ELSE 'Everyone Else'
END As Risk_group, 
Count(*) As Loan_Count,
Round(Count(Case when Status = 1 Then 1 END) / Count (Status) *100,2) As Defaulted_pct
from 
  loan_default_workspace.default.loans_silver
  Group by 1
  order by 3 desc

Risk_group,Loan_Count,Defaulted_pct
Everyone Else,104965,26.54
HIGH DTI + HIGH LTV,43705,20.1


In [0]:
%sql
-- Calculate loan count and defaulted loan percentage by LTV bucket
Select 
CASE
WHEN LTV < 60  THEN 'Low(<60)'
WHEN LTV BETWEEN 60 AND 80 THEN 'Medium(60-80)'
ELSE 'High(>80)'
END As LTV_Bucket, 
Count(*) As Loan_Count,
Round(Count(Case when Status = 1 Then 1 END) / Count (Status) *100,2) As Defaulted_pct
from 
  loan_default_workspace.default.loans_silver
  Group by 1
  order by 3 desc

LTV_Bucket,Loan_Count,Defaulted_pct
Medium(60-80),64993,33.16
High(>80),51112,20.65
Low(<60),32565,13.92


In [0]:
-- Show top 15 LTV values by loan count
SELECT
    LTV,
   count(*) AS loan_count          
FROM loan_default_workspace.default.loans_silver
GROUP BY LTV
ORDER BY loan_count DESC             
LIMIT 15;

LTV,loan_count
75.10245902,15203
81.25,530
91.66666667,499
80.03875969,380
80.03246753,328
94.95614035,322
78.84615385,317
78.64583333,310
80.06329114,309
79.04040404,309


In [0]:
-- Calculate total defaulted loan dollars by region
SELECT Region,
    SUM(CASE
        WHEN Status = 1 THEN loan_amount
        ELSE 0
    END) defaulted_dollars
FROM loan_default_workspace.default.loans_silver
GROUP BY Region
ORDER BY 2 DESC

Region,defaulted_dollars
south,5519375500
North,5346306500
central,719647500
North-East,112594000


In [0]:
-- Show loan count and total defaulted loan dollars by region
SELECT Region, count(loan_amount),
    SUM(CASE
        WHEN Status = 1 THEN loan_amount
        ELSE 0
    END) defaulted_dollars
FROM loan_default_workspace.default.loans_silver
GROUP BY Region
ORDER BY 2 DESC

Region,count(loan_amount),defaulted_dollars
North,74722,5346306500
south,64016,5519375500
central,8697,719647500
North-East,1235,112594000


In [0]:
-- Calculate defaulted loan percentage by region and DTI bucket
%sql
Select 
Region,
  CASE 
    WHEN dtir1 < 30 THEN 'Under 30' 
    WHEN dtir1 >= 30 THEN 'Over 30' 
end AS dti_bucket, 
 COUNT(*) AS loan_count, 
  ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100,2) AS Defaulted_pct
from loan_default_workspace.default.loans_silver
Group by Region,
CASE 
    WHEN dtir1 < 30 THEN 'Under 30' 
    WHEN dtir1 >= 30 THEN 'Over 30' 
end
ORDER BY Defaulted_pct DESC

Region,dti_bucket,Defaulted_pct
North-East,Over 30,31.86
central,Over 30,28.61
south,Over 30,28.22
North,Over 30,24.07
central,Under 30,20.41
North-East,Under 30,19.86
south,Under 30,18.0
North,Under 30,14.71


In [0]:
%sql
CREATE OR REPLACE TABLE loan_default_workspace.default.gold_regional_breakdown AS
SELECT
    Region,
    COUNT(*) AS total_loans,
    COUNT(CASE WHEN Status = 1 THEN 1 END) AS defaulted_loans,
    ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END) / Count(Status) * 100, 2) AS default_rate_pct,
    SUM(CASE WHEN Status = 1 THEN loan_amount ELSE 0 END) AS value_at_risk
FROM loan_default_workspace.default.loans_silver
GROUP BY Region
ORDER BY value_at_risk DESC

Region,total_loans,defaulted_loans,default_rate_pct,value_at_risk
south,64016,17047,26.63,5519375500
North,74722,16821,22.51,5346306500
central,8697,2395,27.54,719647500
North-East,1235,376,30.45,112594000


In [0]:
%sql
SELECT LTV, COUNT(*) AS freq
FROM loans_silver
GROUP BY LTV
ORDER BY freq DESC
LIMIT 20;

LTV,freq
75.10245902,15203
81.25,530
91.66666667,499
80.03875969,380
80.03246753,328
94.95614035,322
78.84615385,317
78.64583333,310
80.06329114,309
79.04040404,309


In [0]:
%sql
SELECT Region, dti_bucket,
  ROUND(SUM(CASE WHEN Status = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct,
  COUNT(*) AS loans
FROM loans_silver
GROUP BY Region, dti_bucket
ORDER BY default_rate_pct DESC;

Region,dti_bucket,default_rate_pct,loans
North-East,Medium,39.48,504
south,Medium,36.55,26846
central,Medium,35.11,3614
North,Medium,32.17,30260
North-East,High,28.10,427
central,High,24.97,2935
south,High,23.00,19182
North,High,19.01,22172
North-East,Low,18.75,304
central,Low,18.30,2148


In [0]:
%sql
SELECT dtir1, COUNT(*) AS freq
FROM loans_silver
GROUP BY dtir1
ORDER BY freq DESC
LIMIT 20;

dtir1,freq
39.0,28661
37.0,6848
36.0,6553
44.0,6500
49.0,6309
43.0,5307
42.0,5121
41.0,4881
40.0,4699
38.0,4461


In [0]:
%sql
SELECT Region, ltv_bucket,
  ROUND(SUM(CASE WHEN Status = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct,
  COUNT(*) AS loans
FROM loans_silver
GROUP BY Region, ltv_bucket
ORDER BY default_rate_pct DESC;

Region,ltv_bucket,default_rate_pct,loans
North-East,Very High,70.83,24
south,Very High,69.93,695
central,Very High,68.60,172
North,Very High,60.57,908
North-East,Medium,37.97,553
central,Medium,35.92,3516
south,Medium,34.78,29038
North,Medium,31.30,31886
North-East,High,24.05,420
south,High,21.71,19426
